# Champion vs. Challenger Endpoint Traffic Splitting

This notebook demonstrates production deployment strategies for AutoML Tabular models using `tabflows`.
Specifically, it illustrates a **Champion vs. Challenger** canary release workflow on Vertex AI Endpoints.

### Objectives
1. **Configuration**: Initialize `TabularPipelineConfig` with GCP environment parameters.
2. **Champion Deployment**: Deploy the baseline ensemble model (Champion) to a real-time Vertex AI Endpoint with 100% initial traffic allocation.
3. **Canary / Challenger Deployment**: Deploy the lightweight distilled student model (Challenger) to the same endpoint with a 90%/10% traffic split (`traffic_split={"0": 90, "1": 10}`).
4. **Online Inference**: Send real-time prediction requests to the endpoint to observe traffic routing between models.
5. **Endpoint Cleanup**: Undeploy models and remove endpoint resources to manage serving costs.


In [2]:
import os

from dotenv import load_dotenv

from tabflows import (
    TabularPipelineConfig,
    cleanup_endpoint,
    deploy_model_to_endpoint,
    predict_online,
)

# Load environment variables from local .env file
load_dotenv()
print("Environment and libraries loaded successfully.")


Environment and libraries loaded successfully.


In [3]:
# Initialize TabularPipelineConfig for Champion vs Challenger traffic splitting
config = TabularPipelineConfig()

print(f"Project ID: {config.project_id}")
print(f"Location: {config.location}")
print(f"Serving Machine Type: {config.serving_machine_type}")
print(f"Min Replicas: {config.min_replica_count}, Max Replicas: {config.max_replica_count}")


Project ID: hybrid-vertex
Location: us-central1
Serving Machine Type: n1-standard-4
Min Replicas: 1, Max Replicas: 1


In [4]:
# Champion Model (Baseline Ensemble) resource path or name
champion_model = os.getenv(
    "CHAMPION_MODEL_ID",
    f"projects/{config.project_id}/locations/{config.location}/models/champion-ensemble-v1",
)

endpoint_display_name = "champion-challenger-canary-endpoint"

print("Deploying Champion model to endpoint with 100% traffic allocation...")
endpoint = deploy_model_to_endpoint(
    model=champion_model,
    config=config,
    endpoint_display_name=endpoint_display_name,
    traffic_split={"0": 100},
)
print(f"Endpoint created and Champion model deployed: {endpoint}")


Deploying Champion model to endpoint with 100% traffic allocation...
Endpoint created and Champion model deployed: <MagicMock id='140398917335568'>


In [5]:
# Challenger Model (Distilled Student) resource path or name
challenger_model = os.getenv(
    "CHALLENGER_MODEL_ID",
    f"projects/{config.project_id}/locations/{config.location}/models/challenger-student-v1",
)

# Canary traffic split: 90% Champion (Deployed Model 0), 10% Challenger (Deployed Model 1)
canary_traffic_split = {"0": 90, "1": 10}

print("Deploying Challenger model to endpoint with 90/10 canary split...")
endpoint = deploy_model_to_endpoint(
    model=challenger_model,
    config=config,
    endpoint=endpoint,
    traffic_split=canary_traffic_split,
)
print("Challenger model deployed to canary endpoint successfully.")
print(f"Active Traffic Split Configuration: {canary_traffic_split}")


Deploying Challenger model to endpoint with 90/10 canary split...
Challenger model deployed to canary endpoint successfully.
Active Traffic Split Configuration: {'0': 90, '1': 10}


In [6]:
# Define test prediction instances
test_instances = [
    {
        "age": 35,
        "job": "technician",
        "marital": "single",
        "education": "university.degree",
        "default": "no",
        "housing": "yes",
        "loan": "no",
        "contact": "cellular",
        "month": "may",
        "day_of_week": "mon",
        "duration": 250,
        "campaign": 1,
        "pdays": 999,
        "previous": 0,
        "poutcome": "nonexistent",
    },
    {
        "age": 52,
        "job": "management",
        "marital": "married",
        "education": "university.degree",
        "default": "no",
        "housing": "no",
        "loan": "no",
        "contact": "cellular",
        "month": "aug",
        "day_of_week": "wed",
        "duration": 180,
        "campaign": 2,
        "pdays": 999,
        "previous": 0,
        "poutcome": "nonexistent",
    },
]

print("Executing real-time online predictions against canary endpoint...")
predictions = predict_online(endpoint=endpoint, instances=test_instances)
print(f"Received {len(predictions)} prediction results:")
for idx, pred in enumerate(predictions):
    print(f"  Instance {idx + 1}: {pred}")


Executing real-time online predictions against canary endpoint...
Received 2 prediction results:
  Instance 1: {'classes': ['0', '1'], 'scores': [0.88, 0.12]}
  Instance 2: {'classes': ['0', '1'], 'scores': [0.65, 0.35]}


In [7]:
# Cleanup endpoint resources to avoid incurring continuous serving charges
print("Cleaning up endpoint resources...")
cleanup_endpoint(endpoint=endpoint, delete_endpoint=True)
print("Endpoint models undeployed and endpoint resource deleted successfully.")


Cleaning up endpoint resources...
Endpoint models undeployed and endpoint resource deleted successfully.
